In [1]:
import os
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [2]:
!pip install pytorch-crf
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=e8a3f3dce923744430b69b5a295da33cebbd34a48b72212f16720ae00c8f612c
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [14]:
import urllib.request

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
from datasets import Dataset, DatasetDict, Features, Sequence, ClassLabel, Value
from seqeval.metrics import classification_report, f1_score

In [15]:
NCBI_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "cambridgeltl/MTL-Bioinformatics-2016/master/data/NCBI-disease-IOB"
)
NCBI_FILES = {"train": "train.tsv", "validation": "devel.tsv", "test": "test.tsv"}
NCBI_LABELS = ["O", "B-Disease", "I-Disease"]


def _parse_iob_tsv(text: str, label2id: dict) -> list:
    sentences, tokens, tags = [], [], []
    for line in text.splitlines():
        line = line.rstrip()
        if not line:
            if tokens:
                sentences.append({"tokens": tokens, "ner_tags": tags})
                tokens, tags = [], []
            continue
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        tok, tag = parts[0], parts[1]
        tokens.append(tok)
        tags.append(label2id.get(tag, 0))
    if tokens:
        sentences.append({"tokens": tokens, "ner_tags": tags})
    return sentences


def load_ncbi_disease() -> DatasetDict:
    label2id = {lab: i for i, lab in enumerate(NCBI_LABELS)}
    features = Features({
        "tokens":   Sequence(Value("string")),
        "ner_tags": Sequence(ClassLabel(names=NCBI_LABELS)),
    })

    splits = {}
    for split, fname in NCBI_FILES.items():
        url = f"{NCBI_BASE_URL}/{fname}"
        print(f"  Downloading {url}")
        with urllib.request.urlopen(url) as r:
            text = r.read().decode("utf-8")
        sents = _parse_iob_tsv(text, label2id)
        print(f"    {split}: {len(sents)} sentences")
        splits[split] = Dataset.from_list(sents, features=features)
    return DatasetDict(splits)

In [17]:
class BioBertCRF(nn.Module):
    """BioBERT encoder + Linear classifier + CRF layer"""

    def __init__(self, model_name: str, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        emissions = self.classifier(sequence_output)

        mask = attention_mask.bool()

        if labels is not None:
            crf_labels = labels.clone()
            crf_labels[crf_labels < 0] = 0
            loss = -self.crf(emissions, crf_labels, mask=mask, reduction="mean")
            return {"loss": loss, "emissions": emissions}

        predictions = self.crf.decode(emissions, mask=mask)
        return {"predictions": predictions, "emissions": emissions}

In [18]:
def tokenize_and_align_labels(examples, tokenizer, max_length: int = 128):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=max_length,
    )
    aligned_labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(0)
            else:
                label_ids.append(label[word_idx])
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized


def collate_fn(batch):
    return {
        "input_ids":      torch.tensor([x["input_ids"]      for x in batch], dtype=torch.long),
        "attention_mask": torch.tensor([x["attention_mask"] for x in batch], dtype=torch.long),
        "labels":         torch.tensor([x["labels"]         for x in batch], dtype=torch.long),
    }

In [19]:
def evaluate(model, loader, label_list, device):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            out = model(input_ids, attention_mask)
            predictions = out["predictions"]

            for i, pred_seq in enumerate(predictions):
                gold = labels[i][attention_mask[i].bool()].tolist()
                all_true.append([label_list[t] for t in gold])
                all_pred.append([label_list[p] for p in pred_seq])

    print(classification_report(all_true, all_pred, digits=4))
    return f1_score(all_true, all_pred)

In [20]:
def train():
    MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
    EPOCHS     = 3
    BATCH_SIZE = 16
    LR_BERT    = 3e-5
    LR_HEAD    = 3e-4
    MAX_LEN    = 128
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print("Loading NCBI Disease dataset (from GitHub raw TSV)...")
    raw = load_ncbi_disease()
    label_list = raw["train"].features["ner_tags"].feature.names
    print(f"Labels: {label_list}")
    num_labels = len(label_list)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    tokenized = raw.map(
        lambda ex: tokenize_and_align_labels(ex, tokenizer, MAX_LEN),
        batched=True,
        remove_columns=raw["train"].column_names,
    )

    train_loader = DataLoader(tokenized["train"],      batch_size=BATCH_SIZE,
                              shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(tokenized["validation"], batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_fn)
    test_loader  = DataLoader(tokenized["test"],       batch_size=BATCH_SIZE,
                              shuffle=False, collate_fn=collate_fn)

    model = BioBertCRF(MODEL_NAME, num_labels=num_labels).to(device)

    bert_params  = [p for n, p in model.named_parameters() if n.startswith("bert.")]
    other_params = [p for n, p in model.named_parameters() if not n.startswith("bert.")]
    optimizer = AdamW([
        {"params": bert_params,  "lr": LR_BERT},
        {"params": other_params, "lr": LR_HEAD},
    ])

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for step, batch in enumerate(train_loader, 1):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            optimizer.zero_grad()
            out  = model(input_ids, attention_mask, labels=labels)
            loss = out["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            if step % 50 == 0:
                print(f"[Epoch {epoch}] step {step}/{len(train_loader)} | loss={loss.item():.4f}")

        print(f"\n[Epoch {epoch}] avg train loss = {total_loss / len(train_loader):.4f}")
        print("--- Validation ---")
        val_f1 = evaluate(model, val_loader, label_list, device)
        print(f"Val F1: {val_f1:.4f}")

    print("\n=========== Test Set ===========")
    test_f1 = evaluate(model, test_loader, label_list, device)
    print(f"Test F1: {test_f1:.4f}")

    torch.save(model.state_dict(), "biobert_crf_ncbi.pt")
    print("Saved: biobert_crf_ncbi.pt")

In [21]:
def predict_sentence(model, tokenizer, sentence: str, label_list, device, max_length: int = 128):
    model.eval()
    words = sentence.split()
    enc = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=max_length,
    ).to(device)

    with torch.no_grad():
        out = model(enc["input_ids"], enc["attention_mask"])
    pred_ids = out["predictions"][0]

    word_ids = enc.word_ids(batch_index=0)
    word2tag = {}
    for tok_idx, w_idx in enumerate(word_ids):
        if w_idx is None or w_idx in word2tag:
            continue
        word2tag[w_idx] = label_list[pred_ids[tok_idx]]

    return [(words[i], word2tag.get(i, "O")) for i in range(len(words))]


if __name__ == "__main__":
    train()


Device: cuda
Loading NCBI Disease dataset (from GitHub raw TSV)...
    train: 5424 sentences
    validation: 923 sentences
    test: 940 sentences
Labels: ['O', 'B-Disease', 'I-Disease']


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5424 [00:00<?, ? examples/s]

Map:   0%|          | 0/923 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: dmis-lab/biobert-base-cased-v1.1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

[Epoch 1] step 50/339 | loss=11.7883
[Epoch 1] step 100/339 | loss=1.8385
[Epoch 1] step 150/339 | loss=3.5052
[Epoch 1] step 200/339 | loss=0.4501
[Epoch 1] step 250/339 | loss=2.1028
[Epoch 1] step 300/339 | loss=1.8139

[Epoch 1] avg train loss = 6.2751
--- Validation ---
              precision    recall  f1-score   support

     Disease     0.8203    0.8628    0.8410      1640

   micro avg     0.8203    0.8628    0.8410      1640
   macro avg     0.8203    0.8628    0.8410      1640
weighted avg     0.8203    0.8628    0.8410      1640

Val F1: 0.8410
[Epoch 2] step 50/339 | loss=0.4660
[Epoch 2] step 100/339 | loss=0.7584
[Epoch 2] step 150/339 | loss=0.5349
[Epoch 2] step 200/339 | loss=1.1347
[Epoch 2] step 250/339 | loss=0.0320
[Epoch 2] step 300/339 | loss=0.0081

[Epoch 2] avg train loss = 1.1699
--- Validation ---
              precision    recall  f1-score   support

     Disease     0.8194    0.8909    0.8536      1640

   micro avg     0.8194    0.8909    0.8536      16